In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder \
    .appName("Banking SWOT ETL") \
    .getOrCreate()

In [2]:
customer_df = spark.read.option("header", True).csv(
    "hdfs://namenode:9000/banking_swot/raw/internal/customer/customer_data.csv"
)

In [3]:
transaction_df = spark.read.option("header", True).csv(
    "hdfs://namenode:9000/banking_swot/raw/internal/transaction/transaction_data.csv"
)

In [4]:
bank_df = spark.read.option("header", True).csv(
    "hdfs://namenode:9000/banking_swot/raw/internal/branch/bank_data.csv"
)

In [5]:
sentiment_df = spark.read.option("header", True).csv(
    "hdfs://namenode:9000/banking_swot/raw/external/sentiment/external_bank_sentiment.csv"
)

In [6]:
transaction_df.show(5)

+--------------+-----------+------------+-------------+------------------+-----------------+---------------+----------------+
|Transaction_ID|Customer_ID|Account_Type|Total_Balance|Transaction_Amount|Investment_Amount|Investment_Type|Transaction_Date|
+--------------+-----------+------------+-------------+------------------+-----------------+---------------+----------------+
|        300000|     209689|    Business|        69339|            4794.0|            42580|  Fixed Deposit|      2024-12-08|
|        300001|     206124|    Business|        12825|            3500.0|            46605|  Fixed Deposit|      2022-09-09|
|        300002|     207501|     Current|        67753|            2401.0|            17027|  Fixed Deposit|      2022-04-28|
|        300003|     208675|     Savings|        67061|            2952.0|             3054|  Fixed Deposit|      2023-05-22|
|        300004|     204923|    Business|         8566|            1025.0|            44937|  Fixed Deposit|      2023

In [ ]:
# Dtata transformation

In [7]:
from pyspark.sql.functions import col, when

transaction_clean = transaction_df \
    .dropDuplicates(["Transaction_ID"]) \
    .withColumnRenamed("Transaction_ID", "transaction_id") \
    .withColumnRenamed("Customer_ID", "customer_id") \
    .withColumnRenamed("Transaction_Amount", "transaction_amount") \
    .filter(col("transaction_amount") > 0)

In [8]:
transaction_clean.show(10)

+--------------+-----------+------------+-------------+------------------+-----------------+-----------------+----------------+
|transaction_id|customer_id|Account_Type|Total_Balance|transaction_amount|Investment_Amount|  Investment_Type|Transaction_Date|
+--------------+-----------+------------+-------------+------------------+-----------------+-----------------+----------------+
|        300000|     209689|    Business|        69339|            4794.0|            42580|    Fixed Deposit|      2024-12-08|
|        300001|     206124|    Business|        12825|            3500.0|            46605|    Fixed Deposit|      2022-09-09|
|        300002|     207501|     Current|        67753|            2401.0|            17027|    Fixed Deposit|      2022-04-28|
|        300003|     208675|     Savings|        67061|            2952.0|             3054|    Fixed Deposit|      2023-05-22|
|        300004|     204923|    Business|         8566|            1025.0|            44937|    Fixed De

In [9]:
sentiment_clean = sentiment_df \
    .dropDuplicates(["post_id"]) \
    .withColumn(
        "sentiment_label",
        when(col("sentiment_score") >= 0.25, "Positive")
        .when(col("sentiment_score") <= -0.25, "Negative")
        .otherwise("Neutral")
    ) \
    .withColumn(
        "swot_signal",
        when(col("sentiment_label") == "Positive", "Opportunity")
        .when(col("sentiment_label") == "Negative", "Threat")
        .otherwise("Monitor")
    )

In [10]:
sentiment_clean.show()

+-------+---------+---------------+------+-------------------+---------------+----------+---------------+-----------+
|post_id|post_date|competitor_bank|region|              topic|sentiment_score|engagement|sentiment_label|swot_signal|
+-------+---------+---------------+------+-------------------+---------------+----------+---------------+-----------+
|      1| 5/1/2024|      HDFC Bank| North|         Mobile App|            0.7|       500|       Positive|Opportunity|
|      2| 6/1/2024|        Maybank| South|   Customer Service|           -0.6|       450|       Negative|     Threat|
|      3| 7/1/2024|           CIMB|  East|    Digital Banking|            0.5|       300|       Positive|Opportunity|
|      4| 8/1/2024|      HDFC Bank|  West|    Online Security|           -0.4|       280|       Negative|     Threat|
|      5| 9/1/2024|        Maybank| North|Investment Products|           0.65|       600|       Positive|Opportunity|
+-------+---------+---------------+------+--------------

In [ ]:
# حفظ البيانات النظيفة

In [12]:
transaction_clean.write.mode("overwrite").parquet(
    "hdfs://namenode:9000/banking_swot/processed/transactions"
)

sentiment_clean.write.mode("overwrite").parquet(
    "hdfs://namenode:9000/banking_swot/processed/sentiment"
)

In [ ]:
# Data Quality Check

In [13]:
from pyspark.sql.functions import col, when, count

# Function to Generate Data Quality Summary


def quality_report(df, dataset_name, key_column):

    raw_rows = df.count()

    duplicate_rows = raw_rows - df.dropDuplicates([key_column]).count()

    # Count missing values in every column
    missing = {}

    for c in df.columns:
        missing_count = df.filter(
            col(c).isNull() | (col(c) == "")
        ).count()

        if missing_count > 0:
            missing[c] = missing_count

    if len(missing) == 0:
        missing_summary = "None"
    else:
        missing_summary = "; ".join(
            [f"{k}: {v}" for k, v in missing.items()]
        )

    clean_rows = df.dropDuplicates([key_column]).count()

    return (
        dataset_name,
        raw_rows,
        duplicate_rows,
        missing_summary,
        clean_rows
    )


# Generate Reports


reports = [

    quality_report(
        customer_df,
        "customer_data.csv",
        "Customer_ID"
    ),

    quality_report(
        transaction_df,
        "transaction_data.csv",
        "Transaction_ID"
    ),

    quality_report(
        bank_df,
        "bank_data.csv",
        "Branch_ID"
    ),

    quality_report(
        sentiment_df,
        "external_bank_sentiment.csv",
        "post_id"
    )

]

# ==========================
# Create Final Summary Table
# ==========================

quality_df = spark.createDataFrame(
    reports,
    [
        "Dataset",
        "Raw Rows",
        "Duplicate Key Rows",
        "Missing Values Found",
        "Clean Rows"
    ]
)

quality_df.show(truncate=False)

+---------------------------+--------+------------------+---------------------------------------+----------+
|Dataset                    |Raw Rows|Duplicate Key Rows|Missing Values Found                   |Clean Rows|
+---------------------------+--------+------------------+---------------------------------------+----------+
|customer_data.csv          |10000   |0                 |Age: 500; Customer_Type: 500; City: 500|10000     |
|transaction_data.csv       |10000   |0                 |None                                   |10000     |
|bank_data.csv              |1000    |0                 |Firm_Revenue: 50                       |1000      |
|external_bank_sentiment.csv|5       |0                 |None                                   |5         |
+---------------------------+--------+------------------+---------------------------------------+----------+

